# Lumen on Kaggle

See `kaggle/README.md` for setup: internet ON, persistence ON (Variables and Files), and a `GITHUB_TOKEN` secret. **Stage 1 stages are CPU-only: set Accelerator to None** so they don't use GPU quota.

In [ ]:
# Clone or update the private repo. The GitHub token never appears in a command line, a URL,
# notebook output, an exception, or on disk: git receives it only as an HTTP auth header through
# environment variables set for the git process alone (tests/test_notebook_clone.py enforces this).
import base64
import os
import subprocess

from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/JCHETAN26/lumen.git"
WORKDIR = os.environ.get("LUMEN_KAGGLE_WORKDIR", "/kaggle/working")


def _git(*args: str) -> None:
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {k: v for k, v in os.environ.items() if not k.startswith("GIT_TRACE")}
    env.update(
        GIT_TERMINAL_PROMPT="0",  # fail instead of prompting for credentials
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
        GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}",
    )
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True, cwd=WORKDIR)
    message = (result.stderr or result.stdout or "").strip()
    for secret in (token, basic):  # redact, in case git ever echoes the header
        message = message.replace(secret, "***")
    del token, basic, env
    if result.returncode != 0:
        raise RuntimeError(
            f"git {args[0]} failed (exit {result.returncode}): {message[-800:]}"
        ) from None


os.makedirs(WORKDIR, exist_ok=True)
if not os.path.isdir(os.path.join(WORKDIR, "lumen")):
    _git("clone", "--depth", "1", REPO_URL, "lumen")
else:
    _git("-C", "lumen", "pull", "--ff-only", REPO_URL, "main")
head = subprocess.run(
    ["git", "-C", os.path.join(WORKDIR, "lumen"), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()
print("repo at", head)

In [ ]:
%cd /kaggle/working/lumen
!bash kaggle/setup.sh

In [ ]:
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py env

## Data (CPU, ~15 GB download, resumable)

Downloads the pinned VizWiz files, verifies them, validates annotations against the image zips, and builds the dev split and Mac sample ID lists.

In [ ]:
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py data

## Mac sample (CPU)

In [ ]:
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py mac_sample

## Stage 2: zero-shot baselines (GPU T4)

Set **Accelerator: GPU T4**, rerun the clone and setup cells, then set `GPU_HOURS_LEFT` from kaggle.com → Settings. Every run refuses to start if its estimate + 1 h reserve exceeds it, and re-checks after a timed calibration batch. Runs resume after a session ends.

In [ ]:
GPU_HOURS_LEFT = 30.0  # <- update from kaggle.com -> Settings before each session

**1. Plan (free):** prints the prior GPU-hour estimate per run; loads nothing.

In [ ]:
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model all --split dev --prompt both --dry-run --gpu-hours-left $GPU_HOURS_LEFT

**2. Pipeline checks (minutes):** a tiny stand-in on the GPU, then 32 real questions with SmolVLM2-500M in fp16 (checks for fp16 NaN/overflow before the long runs). Smoke results go to `baseline/smoke/` and are never mixed with real ones.

In [ ]:
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m lfm2.5-vl-450m --split dev --prompt abstain --stand-in --limit 16
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --limit 32 --gpu-hours-left $GPU_HOURS_LEFT

**3. Dev runs (~5.2 GPU h prior):** both prompts for every model. Choices are made here.

In [ ]:
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model all --split dev --prompt both --gpu-hours-left $GPU_HOURS_LEFT

**4. Val runs (~4.9 GPU h prior):** each model with the prompt chosen on dev. Refuses if a dev result is missing.

In [ ]:
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model all --split val --prompt dev-selected --gpu-hours-left $GPU_HOURS_LEFT

## Download

Download `/kaggle/working/lumen-results/` from the **Output** tab: result JSONs, prediction files, split ID lists, and the Mac sample archive. The image zips stay on Kaggle.